In [1]:
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
import os
from google.cloud import bigquery
from langchain_openai import ChatOpenAI

# Load from .env file
load_dotenv("env.txt")

# Get credentials
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")


# Verify credentials exist
if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY not found in environment")


print("✓ Environment variables loaded")
print(f"  OpenAI API Key: {OPENAI_API_KEY[:20]}...")


✓ Environment variables loaded
  OpenAI API Key: sk-proj-DyCFdOryoZkE...


In [2]:
print("OPENAI_API_KEY exists:", bool(os.getenv("OPENAI_API_KEY")))
print("GOOGLE_APPLICATION_CREDENTIALS:", os.getenv("GOOGLE_APPLICATION_CREDENTIALS"))


OPENAI_API_KEY exists: True
GOOGLE_APPLICATION_CREDENTIALS: None


In [3]:
bq_client = bigquery.Client()
print("Connected to project:", bq_client.project)


Connected to project: project-8661af2e-b3ff-46f2-bc9


In [ ]:
PROJECT = "project-8661af2e-b3ff-46f2-"
DATASET = "RAG"
TABLE = "customers"

table_id = f"{PROJECT}.{DATASET}.{TABLE}"
table = bq_client.get_table(table_id)

schema_text = ""
for field in table.schema:
    schema_text += f"- {field.name} ({field.field_type})"

print(schema_text)


- Index (INTEGER)- Customer_Id (STRING)- First_Name (STRING)- Last_Name (STRING)- Company (STRING)- City (STRING)- Country (STRING)- Phone 1 (STRING)- Phone 2 (STRING)- Email (STRING)- Subscription_Date (DATE)- Website (STRING)


In [5]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)


In [6]:

prompt = ChatPromptTemplate.from_template("""
You are a senior data analyst.
Generate ONLY valid BigQuery SQL.
Do not explain anything.
Do not use markdown.
Only output SQL.

Rules:
- Use fully qualified table name: `{project}.{dataset}.{table}`
- Use BigQuery SQL syntax
- Use LIMIT 100
- Only SELECT queries are allowed

Table schema:
{schema}

User question:
{question}

SQL:
""")


In [7]:
question = "give me count of customers month wise for the year 2022 ?"

In [8]:
chain = prompt | llm

sql = chain.invoke({
    "project": PROJECT,
    "dataset": DATASET,
    "table": TABLE,
    "schema":schema_text,
    "question":question
}).content
print(sql)


SELECT EXTRACT(YEAR FROM Subscription_Date) AS year, 
       EXTRACT(MONTH FROM Subscription_Date) AS month, 
       COUNT(Customer_Id) AS customer_count 
FROM `project-8661af2e-b3ff-46f2-bc9.RAG.customers` 
WHERE EXTRACT(YEAR FROM Subscription_Date) = 2022 
GROUP BY year, month 
ORDER BY month 
LIMIT 100;


In [9]:
#quick safety check(inline)
if not sql.strip().lower().startswith("select"):
    raise ValueError("Unsafe query detected")

In [10]:
#sql query which llm has returned we need to execute the same on bigquery now

query_job = bq_client.query(sql)
results = query_job.result()

rows = [dict(row) for row in results]

print(rows)

[{'year': 2022, 'month': 1, 'customer_count': 31}, {'year': 2022, 'month': 2, 'customer_count': 29}, {'year': 2022, 'month': 3, 'customer_count': 32}, {'year': 2022, 'month': 4, 'customer_count': 51}, {'year': 2022, 'month': 5, 'customer_count': 27}]


In [11]:
explain_prompt =f"""
Explain the following query result in simple businees language.

Result:
{rows}
"""

explanation = llm.invoke(explain_prompt)
print(explanation.content)

The query result shows the number of customers for each month in the first five months of 2022. Here’s a simple breakdown:

- **January 2022**: 31 customers
- **February 2022**: 29 customers
- **March 2022**: 32 customers
- **April 2022**: 51 customers
- **May 2022**: 27 customers

In summary, the customer count varied each month, with the highest number of customers in April (51) and the lowest in February (29). Overall, there was a noticeable increase in customers in April compared to the other months.


In [12]:
## RAG part

In [13]:
from langchain_community.document_loaders import GCSFileLoader
# Change 'langchain.text_splitter' to 'langchain_text_splitters'
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [14]:
from langchain_community.document_loaders import GCSFileLoader
#from langchain-google-community import langchain_google_community 
bucket_name = "minalgenai_staging"
blob_name = "HRPolicy.pdf"

# Capitalize the 'L' in Loader
loader = GCSFileLoader(
    project_name="project-8661af2e-b3ff-46f2-bc9",
    bucket=bucket_name,
    blob=blob_name
)

docs = loader.load()
print(len(docs))

/var/tmp/ipykernel_4122/3850894103.py:7: LangChainDeprecationWarning: The class `GCSFileLoader` was deprecated in LangChain 0.0.32 and will be removed in 1.0. An updated version of the class exists in the `langchain-google-community package and should be used instead. To use it run `pip install -U `langchain-google-community` and import as `from `langchain_google_community import GCSFileLoader``.
  loader = GCSFileLoader(


1


In [15]:
pip install -q "unstructured[all-docs]"

Note: you may need to restart the kernel to use updated packages.


In [15]:
#printing first 100 characters of the document
print(docs[0].page_content[:100])

Indian Industries Association

Human Resource Policy

IIA HR POLICY REVISION 1.1

1

Preface

Indian


In [16]:
#chunck document

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

chunks = text_splitter.split_documents(docs)
len(chunks)
                             

108

In [20]:
#Create embedding using openAI
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

In [21]:
TABLE= "doc_embeddings"
from langchain_community.vectorstores import BigQueryVectorStore

vector_store = BigQueryVectorStore.from_documents(
    documents=chunks,
    project_id=PROJECT,
    dataset_name=DATASET,
    table_name=table,
    location="US"
)

ImportError: cannot import name 'BigQueryVectorStore' from 'langchain_community.vectorstores' (/opt/conda/lib/python3.10/site-packages/langchain_community/vectorstores/__init__.py)

In [34]:
rows = []
import uuid
for doc in chunks:
    rows.append({
        "id": str(uuid.uuid4()),
        "content": doc.page_content,
        "embedding": embeddings.embed_query(doc.page_content),
        "source": doc.metadata.get("source", "gcs")
    })

In [24]:
#IOPub data rate exceeded.
#The Jupyter server will temporarily stop sending output
#to the client in order to avoid crashing it.
#To change this limit, set the config variable
#`--ServerApp.iopub_data_rate_limit`.

#Current values:
#ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
#ServerApp.rate_limit_window=3.0 (secs)

#print(rows)

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [25]:
--ServerApp.iopub_data_rate_limit

NameError: name 'ServerApp' is not defined

In [35]:
rows.append({
    "id": str(uuid.uuid4()),
    "content": doc.page_content,
    "embedding": embeddings.embed_query(doc.page_content),
    "source": doc.metadata.get("source", "gcs")
})


In [36]:
# this step inserts the embedding into bigquery
errors = bq_client.insert_rows_json(
    "project-8661af2e-b3ff-46f2-bc9.RAG.doc_embeddings",
    rows
)
print((errors))

[]


In [38]:

bq_client.query("""
SELECT COUNT(*) AS total_rows
FROM `project-8661af2e-b3ff-46f2-bc9.RAG.doc_embeddings`
""").to_dataframe()


,total_rows
0,109


In [64]:
PROJECT = "project-8661af2e-b3ff-46f2-bc9"
DATASET = "RAG"
TABLE = "doc_embeddings"
LOCATION = "asia-south1"

bq_client = bigquery.Client(
    project=PROJECT,
    location=LOCATION
)

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)


In [65]:
user_question = "HOw MANY LEAVES DO WE GET ANNUALY?"

query_embedding = embeddings.embed_query(user_question)
len(query_embedding)


1536

In [82]:
vector_search_sql = f"""
SELECT
  base.content,
  base.source,
  distance
FROM VECTOR_SEARCH(
  TABLE `{PROJECT}.{DATASET}.{TABLE}`,
  'embedding',
  (SELECT @query_embedding AS embedding),
  top_k => 5
)
ORDER BY distance
"""

In [83]:
from google.cloud import bigquery

job_config = bigquery.QueryJobConfig(
    query_parameters=[
        bigquery.ArrayQueryParameter(
            "query_embedding",
            "FLOAT64",
            query_embedding
        )
    ]
)

results = bq_client.query(
    vector_search_sql,
    job_config=job_config
).result()

docs = list(results)
len(docs)


5

In [84]:
len(query_embedding)


1536

In [85]:
# Extract the text content from your BigQuery results
context_text = "\n\n".join([row.content for row in docs])

print("--- Context Preview ---")
print(context_text[:500] + "...")

--- Context Preview ---
will not be treated as a part of casual leave. Casual leaves not availed during the year will

not be carried forward to next calendar year.

✓ Earned Leave: After completion of 1 year in IIA employee will be given 15 earned leave.

Earned leaves will be credited in January each year for the previous year proportionate to

regular attendance in the Office excluding un-authorised absence / leave without pay

fractions rounded off to 1 if >or= 0.5 and 0 if < 0.5.

✓ Maternity Leave as per Law.

✓ ...
